In [1]:
"""
AgroSense — Live IoT Stream Simulator (BigQuery FREE Version)
=============================================================
Uses load_table_from_json() .
This is 100% free — no billing required.
Inserts a batch of rows every 30 seconds into BigQuery.

Install:
    pip install google-cloud-bigquery pandas numpy schedule

Run:
    python scheduler_bigquery.py
"""

import schedule
import time
import signal
import sys
import pandas as pd
import numpy as np
from datetime import datetime, UTC, timezone
from google.cloud import bigquery
from google.oauth2 import service_account

# ── CONFIG — update these values ──────────────────────────
PROJECT_ID   = "agrosense-493415"          # e.g. agrosense-project-123
DATASET_ID   = "agrosense"        # your BigQuery dataset name
TABLE_ID     = "fact_sensor_readings"     # your BigQuery table name
KEY_FILE     = "agrosense_key.json"       # path to your downloaded JSON key
YIELD_CSV    = "./cleaned_data/cleaned_smart_farming_df_new.csv"
STREAM_EVERY = 3                         # seconds between each insert
# ──────────────────────────────────────────────────────────


BATCH_SIZE   = 5    # rows to insert per run (5 rows every 30 seconds)
RUN_EVERY    = 30 # seconds between each batch insert
# ──────────────────────────────────────────────────────────
 
FULL_TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"
 
print("\n" + "="*60)
print("  AgroSense — BigQuery FREE TIER Scheduler")
print(f"  File     : scheduler_bigquery_free.py")
print(f"  Method   : load_table_from_json ONLY (free tier)")
print(f"  Table    : {FULL_TABLE_ID}")
print(f"  Batch    : {BATCH_SIZE} rows every {RUN_EVERY}s")
print("="*60)
 
 
# ── STEP 3: AUTHENTICATE TO BIGQUERY ──────────────────────
print("\n[INIT] Authenticating to BigQuery...")
 
try:
    credentials = service_account.Credentials.from_service_account_file(
        KEY_FILE,
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    client = bigquery.Client(
        credentials=credentials,
        project=PROJECT_ID
    )
    # Quick test — list tables to confirm connection works
    tables = list(client.list_tables(f"{PROJECT_ID}.{DATASET_ID}"))
    print(f"  ✓ Connected to BigQuery project: {PROJECT_ID}")
    print(f"  ✓ Dataset '{DATASET_ID}' has {len(tables)} tables")
except FileNotFoundError:
    print(f"  ✗ Key file not found: {KEY_FILE}")
    print("  Check the KEY_FILE path in CONFIG section above.")
    sys.exit(1)
except Exception as e:
    print(f"  ✗ Authentication failed: {e}")
    sys.exit(1)
 
 
# ── STEP 4: LOAD CSV ───────────────────────────────────────
print("\n[INIT] Loading CSV...")
 
try:
    df = pd.read_csv(YIELD_CSV)
    print(f"  ✓ CSV loaded: {len(df)} rows × {len(df.columns)} columns")
except FileNotFoundError:
    print(f"  ✗ CSV not found: {YIELD_CSV}")
    print("  Check the YIELD_CSV path in CONFIG section above.")
    sys.exit(1)
 
 
# ── STEP 5: LOOKUP MAPS ────────────────────────────────────
REGION_ID_MAP = {
    "North India": 1,
    "South India": 2,
    "Central USA": 3,
    "South USA"  : 4,
    "East Africa": 5,
}
 

 
# Counters
pointer    = [0]   # which CSV row to read next
row_count  = [0]   # total rows inserted this session
batch_num  = [0]   # total batches run
 
 
# ── STEP 6: HELPER FUNCTIONS ───────────────────────────────
def add_noise(value, std, lo, hi):
    """Add Gaussian noise to a sensor value and clamp to range."""
    noisy = float(value) + np.random.normal(0, std)
    return round(float(np.clip(noisy, lo, hi)), 2)
 
def compute_et0(temp, sunlight):
    """Hargreaves simplified ET0 formula (FAO-56)."""
    return round(0.0023 * (float(temp) + 17.8) * (float(sunlight) ** 0.5) * 0.408, 3)
 
def detect_anomaly(moisture, temp, humidity):
    """IQR threshold anomaly detection."""
    am = 1 if (moisture < 12.0 or moisture > 43.0) else 0
    at = 1 if (temp     < 16.0 or temp     > 33.0) else 0
    ah = 1 if (humidity < 42.0 or humidity > 88.0) else 0
    return am, at, ah, max(am, at, ah)
 
def utc_now_str():
    """
    Current UTC timestamp as string.
    Uses timezone-aware datetime — no deprecation warning.
    """
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
 
def build_one_record(row, record_id):
    """
    Build a single sensor reading dict from one CSV row.
    Adds sensor noise, computes ET0, detects anomalies.
    """
    moisture  = add_noise(row["soil_moisture_%"],    0.8,  10.0, 45.0)
    temp      = add_noise(row["temperature_C"],      0.5,  15.0, 35.0)
    humidity  = add_noise(row["humidity_%"],         1.2,  40.0, 90.0)
    ph        = add_noise(row["soil_pH"],            0.02,  5.5,  7.5)
    ndvi      = add_noise(row["NDVI_index"],         0.005, 0.30, 0.90)
    sunlight  = round(float(row["sunlight_hours"]),    2)
    rainfall  = round(float(row["rainfall_mm"]),       2)
    pesticide = round(float(row["pesticide_usage_ml"]),2)
    et0       = compute_et0(temp, sunlight)
    am, at, ah, af = detect_anomaly(moisture, temp, humidity)
    ts        = utc_now_str()
 
    return {
        "reading_id"         : int(record_id),
        "farm_id"            : str(row["farm_id"]),
        "sensor_id"          : str(row["sensor_id"]),
        "crop_type"          : str(row["crop_type"]),
        "region_id"          : int(REGION_ID_MAP.get(str(row["region"]), 0)),
        "region_name"        : str(row["region"]),
        "reading_ts"         : ts,
        "soil_moisture_pct"  : moisture,
        "soil_pH"            : ph,
        "temperature_C"      : temp,
        "humidity_pct"       : humidity,
        "rainfall_mm"        : rainfall,
        "sunlight_hours"     : sunlight,
        "pesticide_usage_ml" : pesticide,
        "NDVI_index"         : ndvi,
        "ET0_computed"       : et0,
        "anomaly_moisture"   : int(am),
        "anomaly_temp"       : int(at),
        "anomaly_humidity"   : int(ah),
        "anomaly_flag"       : int(af),
        "ingested_at"        : ts,
    }
 
 
# ── STEP 7: BATCH INSERT FUNCTION (FREE TIER) ──────────────
def run_batch():
    """
    ══════════════════════════════════════════════════
    THIS IS THE ONLY INSERT METHOD IN THIS FILE.
    Uses: client.load_table_from_json()
    Does NOT use: client.insert_rows_json()
    ══════════════════════════════════════════════════
    """
    batch_num[0] += 1
    records = []
 
    # Build BATCH_SIZE records
    for _ in range(BATCH_SIZE):
        idx = pointer[0] % len(df)
        row = df.iloc[idx]
        pointer[0]  += 1
        row_count[0] += 1
        records.append(build_one_record(row, row_count[0]))
 
    # BigQuery job configuration
    job_config = bigquery.LoadJobConfig(
        # Append to existing rows — never overwrite
        write_disposition     = bigquery.WriteDisposition.WRITE_APPEND,
        # Input format is list of JSON dicts
        source_format         = bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
        # Skip rows with fields not in schema (safety net)
        ignore_unknown_values = True,
        # Skip rows with type mismatches instead of failing
        max_bad_records       = 2,
    )
 
    print(f"\n[{datetime.now().strftime('%H:%M:%S')}] "
          f"Batch #{batch_num[0]} — inserting {BATCH_SIZE} rows...")
 
    try:
        # ════════════════════════════════════════════════
        # FREE TIER INSERT — load_table_from_json()
        # This is a Load Job, not a Streaming Insert.
        # Load Jobs are free on BigQuery free tier.
        # ════════════════════════════════════════════════
        job = client.load_table_from_json(
            json_rows     = records,          # list of dicts
            destination   = FULL_TABLE_ID,    # project.dataset.table
            job_config    = job_config,
        )
 
        # Wait for the load job to finish (usually 3-8 seconds)
        print(f"  ⏳ Job submitted: {job.job_id}")
        print(f"  ⏳ Waiting for job to complete...")
        job.result()   # blocks until done
 
        # Check for errors returned by the job
        if job.errors:
            print(f"  ✗ Job completed with errors:")
            for err in job.errors:
                print(f"     {err}")
            return
 
        # ── Success ──────────────────────────────────
        anomalies = sum(r["anomaly_flag"] for r in records)
        loops     = pointer[0] // len(df) + 1
 
        print(f"  ✓ Inserted {len(records)} rows successfully")
        print(f"  ✓ Total rows this session : {row_count[0]}")
        print(f"  ✓ Total anomalies         : {anomalies} in this batch")
        print(f"  ✓ CSV loop number         : {loops}")
        print(f"  ─────────────────────────────────────────────")
 
        for r in records:
            status = "⚠ ANOMALY" if r["anomaly_flag"] else "✓ Normal"
            print(
                f"  → {r['region_name']:<15} | "
                f"{r['crop_type']:<8} | "
                f"Moisture:{r['soil_moisture_pct']:>5}% | "
                f"Temp:{r['temperature_C']:>5}°C | "
                f"NDVI:{r['NDVI_index']:.3f} | "
                f"{status}"
            )
 
        print(f"\n  Next batch in {RUN_EVERY} seconds... (Ctrl+C to stop)")
 
    # ── ERROR HANDLING ────────────────────────────────
    except Exception as e:
        err = str(e)
        print(f"\n  ════ INSERT ERROR ════")
 
        if "insertAll" in err or "Streaming insert" in err:
            print("  ✗ STREAMING API CALLED — you are running the OLD file")
            print("  ✗ Make sure you are running: scheduler_bigquery_free.py")
            print("  ✗ NOT the old scheduler.py")
            print()
            print("  To check which file is running:")
            print("  Windows: tasklist | findstr python")
            print("  Mac/Linux: ps aux | grep python")
 
        elif "404" in err:
            print(f"  ✗ Table not found: {FULL_TABLE_ID}")
            print("  Check PROJECT_ID, DATASET_ID, TABLE_ID in CONFIG")
 
        elif "403" in err:
            print("  ✗ Permission denied")
            print("  Your service account needs BigQuery Admin role")
            print("  Go to: console.cloud.google.com → IAM & Admin")
 
        elif "schema" in err.lower() or "invalid" in err.lower():
            print(f"  ✗ Schema error: {err}")
            print("  A column name in build_one_record() doesn't match BigQuery")
 
        elif "json" in err.lower():
            print(f"  ✗ JSON error: {err}")
            print("  A value type is wrong — check int/float/string types")
 
        else:
            print(f"  ✗ Unknown error: {err}")
 
        print(f"  ════════════════════")
        print(f"  Will retry next batch in {RUN_EVERY} seconds...\n")
 
 
# ── STEP 8: VERIFY TABLE EXISTS BEFORE STARTING ────────────
print("\n[INIT] Verifying BigQuery table exists...")
try:
    table = client.get_table(FULL_TABLE_ID)
    print(f"  ✓ Table found: {FULL_TABLE_ID}")
    print(f"  ✓ Current row count: {table.num_rows}")
    print(f"  ✓ Table type: {table.table_type}")
 
    if table.table_type == "EXTERNAL":
        print()
        print("  ✗ CRITICAL: Table type is EXTERNAL (Google Sheets link)")
        print("  ✗ External tables are READ-ONLY — inserts will always fail")
        print("  ✗ You must delete this table and recreate it as a native table")
        print("  ✗ See: bigquery console → delete table → recreate with schema")
        sys.exit(1)
 
except Exception as e:
    print(f"  ✗ Table not found or access error: {e}")
    print(f"  Create the table first using agrosense_bigquery_schema.sql")
    sys.exit(1)
 
 
# ── STEP 9: GRACEFUL SHUTDOWN ──────────────────────────────
def shutdown(sig, frame):
    print(f"\n\n{'='*60}")
    print(f"  ✓ AgroSense stream stopped cleanly.")
    print(f"  Rows inserted : {row_count[0]}")
    print(f"  Batches run   : {batch_num[0]}")
    print(f"  CSV loops     : {pointer[0] // len(df)}")
    print(f"\n  Verify in BigQuery:")
    print(f"  SELECT COUNT(*) FROM `{FULL_TABLE_ID}`")
    print(f"{'='*60}\n")
    sys.exit(0)
 
signal.signal(signal.SIGINT,  shutdown)
signal.signal(signal.SIGTERM, shutdown)
 
 
# ── STEP 10: START STREAMING ───────────────────────────────
print("\n[START] Running first batch immediately...")
run_batch()
 
print(f"\n[START] Scheduling batch every {RUN_EVERY} seconds...")
schedule.every(RUN_EVERY).seconds.do(run_batch)
 
while True:
    try:
        schedule.run_pending()
        time.sleep(1)
    except KeyboardInterrupt:
        shutdown(None, None)
 
 
# ══════════════════════════════════════════════════════════
# TROUBLESHOOTING
# ══════════════════════════════════════════════════════════
#
# ERROR: 403 Streaming insert is not allowed in the free tier
# ─────────────────────────────────────────────────────────
# This means insert_rows_json() was called somewhere.
# This file does NOT use insert_rows_json() at all.
# Solution:
#   1. Make sure you are running THIS file, not the old one
#   2. Run: python scheduler_bigquery_free.py
#   3. NOT: python scheduler.py  (old file)
#
# ERROR: Table type is EXTERNAL
# ─────────────────────────────────────────────────────────
# Your table is linked to Google Sheets — it's read-only.
# Solution:
#   1. BigQuery Console → delete fact_sensor_readings
#   2. Recreate as native table using agrosense_bigquery_schema.sql
#   3. Run one_time_load.py to reload your 500 rows
#   4. Then run this scheduler
#
# ERROR: Table not found
# ─────────────────────────────────────────────────────────
# Solution: Check PROJECT_ID, DATASET_ID, TABLE_ID in CONFIG
#
# ERROR: Job completed with errors (schema mismatch)
# ─────────────────────────────────────────────────────────
# A column name in build_one_record() doesn't match BigQuery.
# Solution: BigQuery Console → your table → Schema tab
# Compare every column name exactly with build_one_record()
# ══════════════════════════════════════════════════════════


  AgroSense — BigQuery FREE TIER Scheduler
  File     : scheduler_bigquery_free.py
  Method   : load_table_from_json ONLY (free tier)
  Table    : agrosense-493415.agrosense.fact_sensor_readings
  Batch    : 5 rows every 30s

[INIT] Authenticating to BigQuery...
  ✓ Connected to BigQuery project: agrosense-493415
  ✓ Dataset 'agrosense' has 15 tables

[INIT] Loading CSV...
  ✓ CSV loaded: 500 rows × 22 columns

[INIT] Verifying BigQuery table exists...
  ✓ Table found: agrosense-493415.agrosense.fact_sensor_readings
  ✓ Current row count: 1640
  ✓ Table type: TABLE

[START] Running first batch immediately...

[10:16:39] Batch #1 — inserting 5 rows...
  ⏳ Job submitted: deb68161-937b-4a9c-b524-1f62a38642f8
  ⏳ Waiting for job to complete...
  ✓ Inserted 5 rows successfully
  ✓ Total rows this session : 5
  ✓ Total anomalies         : 2 in this batch
  ✓ CSV loop number         : 1
  ─────────────────────────────────────────────
  → North India     | Wheat    | Moisture:35.34% | Temp: 1

SystemExit: 0

C:\Users\manuj\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
